In [ ]:
# Cell 8: side-by-side comparison table, DistilBERT vs baseline

baseline_df = pd.read_csv(BASELINE_PATH)
baseline_map = {}
for _, r in baseline_df.iterrows():
    baseline_map[(r['attack'], r['intensity'])] = r

print("=" * 95)
print("DISTILBERT vs BASELINE: adversarial evaluation comparison")
print("=" * 95)
print(f"{'Attack':<12} {'Intensity':<10} {'BL Acc':>8} {'BL F1':>8} {'DB Acc':>8} {'DB F1':>8} {'Acc Diff':>9} {'F1 Diff':>9}")
print("-" * 95)

for r in results:
    key = (r['attack'], r['intensity'])
    bl = baseline_map[key]
    acc_diff = r['accuracy'] - bl['accuracy']
    f1_diff = r['f1'] - bl['f1']
    print(f"{r['attack']:<12} {r['intensity']:<10} {bl['accuracy']:>8.4f} {bl['f1']:>8.4f} {r['accuracy']:>8.4f} {r['f1']:>8.4f} {acc_diff:>+9.4f} {f1_diff:>+9.4f}")

print("-" * 95)

# Degradation from each model's own clean baseline
db_clean = results[0]
bl_clean = baseline_map[('clean', 'none')]

print(f"\n{'Attack':<12} {'Intensity':<10} {'BL Acc Drop':>12} {'BL F1 Drop':>12} {'DB Acc Drop':>12} {'DB F1 Drop':>12}")
print("-" * 70)
for r in results[1:]:
    key = (r['attack'], r['intensity'])
    bl = baseline_map[key]
    bl_ad = bl_clean['accuracy'] - bl['accuracy']
    bl_fd = bl_clean['f1'] - bl['f1']
    db_ad = db_clean['accuracy'] - r['accuracy']
    db_fd = db_clean['f1'] - r['f1']
    print(f"{r['attack']:<12} {r['intensity']:<10} {bl_ad:>+12.4f} {bl_fd:>+12.4f} {db_ad:>+12.4f} {db_fd:>+12.4f}")
print("-" * 70)
print("\nBL = baseline (TF-IDF + LR)    DB = DistilBERT")

In [ ]:
# Cell 7: save the per-set metrics to a CSV

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_PATH, index=False)
print(f"Results saved to: {OUTPUT_PATH}")
results_df

In [ ]:
# Cell 6: run every evaluation in fixed order, clean first

results = []

# 1. Clean test set, the anchor we'll measure drops from
print("Evaluating clean test set...")
row = evaluate(clean_test['text'], clean_test['label'], 'clean', 'none')
results.append(row)
print(f"  Accuracy: {row['accuracy']}  F1: {row['f1']}")

# 2. Every adversarial set, in a fixed order so the output CSV is stable
adversarial_files = [
    ('char', 'light'),
    ('char', 'medium'),
    ('char', 'heavy'),
    ('synonym', 'light'),
    ('synonym', 'medium'),
    ('synonym', 'heavy'),
    ('dilution', 'light'),
    ('dilution', 'medium'),
    ('dilution', 'heavy'),
]

for attack, intensity in adversarial_files:
    filename = f'test_{attack}_{intensity}.csv'
    filepath = os.path.join(ATTACKS_DIR, filename)
    print(f"Evaluating {filename}...")

    adv_df = pd.read_csv(filepath)
    row = evaluate(adv_df['text'], adv_df['label'], attack, intensity)
    results.append(row)
    print(f"  Accuracy: {row['accuracy']}  F1: {row['f1']}")

print("\nAll evaluations complete.")

In [ ]:
# Cell 5: predict + evaluate helpers

# 128 is fine on a Colab T4 with FP32. 512 max length so dilution-attack
# rows (which add words) don't get truncated harder than the clean set.
BATCH_SIZE = 128
MAX_LENGTH = 512

def predict_batch(texts):
    """Tokenise and forward each chunk; return (predictions, spam_probabilities)."""
    all_preds = []
    all_probs = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i + BATCH_SIZE]
        encodings = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        )
        encodings = {k: v.to(device) for k, v in encodings.items()}

        with torch.no_grad():
            outputs = model(**encodings)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        spam_probs = probs[:, 1].cpu().numpy()

        all_preds.extend(preds)
        all_probs.extend(spam_probs)

    return np.array(all_preds), np.array(all_probs)


def evaluate(texts, labels, attack_name, intensity):
    """Predict on (texts, labels) and return one row of every metric."""
    text_list = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
    y_pred, y_prob = predict_batch(text_list)
    y_true = np.array(labels)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        'attack': attack_name,
        'intensity': intensity,
        'accuracy': round(accuracy_score(y_true, y_pred), 4),
        'precision': round(precision_score(y_true, y_pred), 4),
        'recall': round(recall_score(y_true, y_pred), 4),
        'f1': round(f1_score(y_true, y_pred), 4),
        'roc_auc': round(roc_auc_score(y_true, y_prob), 4),
        'true_negatives': int(tn),
        'false_positives': int(fp),
        'false_negatives': int(fn),
        'true_positives': int(tp),
    }

print("Evaluation functions defined.")

In [ ]:
# Cell 4: rebuild the clean test set, same split as training

print("Reproducing clean test set...")
df_raw = load_enron_data(DATA_PATH)
df = preprocess_dataset(df_raw)
_, X_test, _, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)
clean_test = pd.DataFrame({'text': X_test.values, 'label': y_test.values})
print(f"Clean test set: {len(clean_test):,} rows (Ham: {(clean_test['label']==0).sum()}, Spam: {(clean_test['label']==1).sum()})")

In [ ]:
# Cell 3: preprocessing functions, copied verbatim from ml/preprocessing.py.
# Inlined because Colab can't see the project tree, and uploading the
# file separately is one more step to forget. If ml/preprocessing.py
# changes, this MUST change too, otherwise inference text drifts from
# training text and results stop being meaningful.

def load_enron_data(filepath):
    csv.field_size_limit(10 ** 7)
    df = pd.read_csv(filepath, encoding='utf-8', on_bad_lines='skip')
    return df

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(
        r'^(from|to|cc|bcc|subject|date|sent|received|reply-to|'
        r'content-type|mime-version|x-mailer|x-originating-ip):.*$',
        '', text, flags=re.MULTILINE
    )
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def combine_subject_message(row):
    subject = clean_text(row.get('Subject', ''))
    message = clean_text(row.get('Message', ''))
    return f"{subject} {message}".strip()

def preprocess_dataset(df):
    df = df.copy()
    df['text'] = df.apply(combine_subject_message, axis=1)
    df['label'] = (df['Spam/Ham'].str.strip().str.lower() == 'spam').astype(int)
    df = df[df['text'].str.len() > 0].reset_index(drop=True)
    return df[['text', 'label']]

print("Preprocessing functions defined.")

In [ ]:
# Cell 2: imports and model loading
import csv
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.to(device)
model.eval()
print(f"Model loaded: {type(model).__name__}")

In [ ]:
# Cell 1: mount Google Drive and resolve every path used downstream
from google.colab import drive
drive.mount('/content/drive')

import os

# Adjust this if the project lives somewhere else on Drive
PROJECT_DIR = '/content/drive/MyDrive/mini-mail'

MODEL_DIR = os.path.join(PROJECT_DIR, 'ml', 'models', 'distilbert_spam_classifier')
DATA_PATH = os.path.join(PROJECT_DIR, 'ml', 'data', 'raw', 'enron_spam_data.csv')
ATTACKS_DIR = os.path.join(PROJECT_DIR, 'attacks')
RESULTS_DIR = os.path.join(ATTACKS_DIR, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
BASELINE_PATH = os.path.join(RESULTS_DIR, 'baseline_adversarial_results.csv')
OUTPUT_PATH = os.path.join(RESULTS_DIR, 'distilbert_adversarial_results.csv')

# Fail fast if any expected path is missing rather than discovering
# halfway through inference that one of these was wrong.
for p in [MODEL_DIR, DATA_PATH, ATTACKS_DIR, BASELINE_PATH]:
    assert os.path.exists(p), f"Missing: {p}"
print("All paths verified.")

# Evaluate DistilBERT against the adversarial test sets

**Author:** Abdulla AlBassam | **Module:** KV6013, Individual Computing Project

The Colab counterpart of `attacks/evaluate_distilbert.py`. Runs the fine-tuned DistilBERT against the nine adversarial CSVs plus the clean test set, then prints a side-by-side comparison against the baseline numbers. Hosted here because the laptop's CPU runs this in roughly an hour, while a Colab T4 finishes it in a couple of minutes.

**Setup:** upload your `mini-mail` folder to Google Drive at
`My Drive/mini-mail/` (or change `PROJECT_DIR` in Cell 1).